# nb04 — Micro Scale: ESP32, MCU, RPi Zero
> **Smallest possible models: from 241 parameters to 50 K.**  
> Train all MCU-class tiers, audit FLASH/RAM budgets, and export a C header
> that drops straight into ESP-IDF or Arduino firmware.

---

## Target hardware and size budgets

| Tier | Extractor | Head | Params | ONNX (fp32) | int8 est. | Target |
|------|-----------|------|--------|-------------|-----------|--------|
| `esp32_nano` | MFCC-13 | FFN-16 | ~241 | <5 KB | <2 KB | ESP32 sub-1 KB weights |
| `esp32_sweet` | MFCC-13 | FFN-64 | ~1 K | ~10 KB | ~3 KB | ESP32 sub-10 KB |
| `esp32_max` | MFCC-13 | FFN-128 | ~2 K | ~15 KB | ~5 KB | ESP32 sub-50 KB |
| `micro` | MFCC-40 | FFN-128 | ~50 K | ~200 KB | ~50 KB | MCU / RPi Zero |
| `delta_micro` | delta-MFCC-13 | FFN-128 | ~55 K | ~220 KB | ~55 KB | MCU / RPi Zero |

**ESP32 constraints:**
- IRAM: 328 KB total, ~150 KB usable for code + weights
- PSRAM (WROVER): 4 MB — enables `micro` tier
- Without PSRAM, stick to `esp32_nano` or `esp32_sweet`

**RPi Zero 2W constraints:**
- 512 MB RAM — any tier fits comfortably
- Cortex-A53 × 4 — `micro` tier runs at ~3 ms/frame

---

## Outputs

```
ww_output/
├── models/<tier>/best_f1.onnx
├── models/<tier>/best_f1_featurizer.onnx
├── esp32_export/model.h          # C header for best ESP32 tier
└── micro_results.csv             # summary table
```

In [ ]:
import os

# ── Core ──────────────────────────────────────────────────────────────────────
WAKE_WORD        = os.environ.get("WAKE_WORD",        "hey jarvis")
OUTPUT_DIR       = os.environ.get("OUTPUT_DIR",       "./ww_output")
DEVICE           = os.environ.get("DEVICE",           "auto")
SEED             = int(os.environ.get("SEED",         "42"))

# ── Model ─────────────────────────────────────────────────────────────────────
# TIER controls which tier to export as C header at the end
TIER             = os.environ.get("TIER",             "esp32_sweet")
EPOCHS           = int(os.environ.get("EPOCHS",       "20"))
BATCH_SIZE       = int(os.environ.get("BATCH_SIZE",   "16"))

# ── Dataset ───────────────────────────────────────────────────────────────────
N_POSITIVE       = int(os.environ.get("N_POSITIVE",   "300"))
LANG             = os.environ.get("LANG",             "en")
ADVERSARIAL      = os.environ.get("ADVERSARIAL",      "true").lower() == "true"
DOWNLOAD_AUGMENT = os.environ.get("DOWNLOAD_AUGMENT", "true").lower() == "true"
REUSE_DATASET    = os.environ.get("REUSE_DATASET",    "true").lower() == "true"
CUSTOM_TRAIN_CSV = os.environ.get("CUSTOM_TRAIN_CSV", "")
CUSTOM_TEST_CSV  = os.environ.get("CUSTOM_TEST_CSV",  "")

# ── Micro-specific ────────────────────────────────────────────────────────────
# SIZE_BUDGET_KB: warn if any ONNX head file exceeds this
SIZE_BUDGET_KB   = int(os.environ.get("SIZE_BUDGET_KB", "50"))
SKIP_COMPLETED   = os.environ.get("SKIP_COMPLETED",   "true").lower() == "true"

# ── MLflow (optional) ────────────────────────────────────────────────────────
MLFLOW_URI       = os.environ.get("MLFLOW_URI",       "")
MLFLOW_SECRET    = os.environ.get("MLFLOW_SECRET",    "MLFLOW_TOKEN")

MICRO_TIERS = ["esp32_nano", "esp32_sweet", "esp32_max", "micro", "delta_micro"]
print(f"Wake word : {WAKE_WORD!r}")
print(f"Tiers     : {MICRO_TIERS}")
print(f"Epochs    : {EPOCHS}  |  Batch size: {BATCH_SIZE}")
print(f"C-export  : {TIER!r} (best ESP32 tier)")

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "pandas", "librosa", "onnx", "onnxruntime", "click", "tqdm")
_pip("ovos-plugin-manager", "ovos-tts-plugin-edge-tts",
     "ovos-vad-plugin-silero", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

import os
_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)

import torch
torch.set_num_threads(min(12, os.cpu_count() or 4))
os.environ.setdefault("OMP_NUM_THREADS", str(min(12, os.cpu_count() or 4)))

print(f"Platform : {_platform}")
print(f"CPU cores: {os.cpu_count()}  |  torch threads: {torch.get_num_threads()}")
print(f"CUDA     : {torch.cuda.is_available()}")

In [ ]:
import shutil
from pathlib import Path

# Disk space guard
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > 3, f"Only {free_gb:.1f} GB free — need at least 3 GB."
print(f"Disk free: {free_gb:.1f} GB")

_aug_kwargs_full = {}

if CUSTOM_TRAIN_CSV:
    import random, csv as _csv
    from ww_trainer.utils import read_dataset_csv
    train_csv = Path(CUSTOM_TRAIN_CSV)
    if CUSTOM_TEST_CSV:
        test_csv = Path(CUSTOM_TEST_CSV)
    else:
        split_dir = Path(OUTPUT_DIR) / "dataset_split"
        split_dir.mkdir(parents=True, exist_ok=True)
        split_train = split_dir / "train.csv"
        split_test  = split_dir / "test.csv"
        if not split_train.exists():
            rows = read_dataset_csv(train_csv)
            random.seed(SEED)
            random.shuffle(rows)
            cut = int(len(rows) * 0.8)
            for path, rs in [(split_train, rows[:cut]), (split_test, rows[cut:])]:
                with open(path, "w", newline="") as f:
                    _csv.writer(f).writerows(rs)
        train_csv, test_csv = split_train, split_test
    print(f"BYO mode: train={train_csv}")
else:
    from ww_trainer.datagen import DatagenConfig, run_datagen_pipeline, DatagenResult, normalize_wake_word
    dataset_dir = Path(OUTPUT_DIR) / "dataset"
    _train_csv_check = dataset_dir / "train" / "metadata.csv"
    if REUSE_DATASET and _train_csv_check.exists():
        print(f"Reusing dataset at {dataset_dir}")
        slug = normalize_wake_word(WAKE_WORD)
        _dr = DatagenResult(
            train_csv=dataset_dir / "train" / "metadata.csv",
            test_csv=dataset_dir / "test" / "metadata.csv",
            positives_dir=dataset_dir / slug / "positives",
            negatives_dir=dataset_dir / slug / "negatives",
            bg_noise_dir=dataset_dir / "augmentation" / "bg_noise",
            music_dir=dataset_dir / "augmentation" / "music",
            rir_dir=dataset_dir / "augmentation" / "rir",
        )
    else:
        print(f"Running datagen for '{WAKE_WORD}'...")
        _dr = run_datagen_pipeline(DatagenConfig(
            wake_word=WAKE_WORD,
            output_dir=dataset_dir,
            n_positive=N_POSITIVE,
            lang=LANG,
            adversarial=ADVERSARIAL,
            vad_trim=True,
            download_augmentation=DOWNLOAD_AUGMENT,
            seed=SEED,
        ))
    train_csv = _dr.train_csv
    test_csv  = _dr.test_csv
    if hasattr(_dr, "bg_noise_dir") and _dr.bg_noise_dir and Path(_dr.bg_noise_dir).exists():
        _aug_kwargs_full["bg_noise_folder"] = str(_dr.bg_noise_dir)
    if hasattr(_dr, "music_dir") and _dr.music_dir and Path(_dr.music_dir).exists():
        _aug_kwargs_full["music_folder"] = str(_dr.music_dir)
    if hasattr(_dr, "rir_dir") and _dr.rir_dir and Path(_dr.rir_dir).exists():
        _aug_kwargs_full["rir_folder"] = str(_dr.rir_dir)

print(f"train_csv: {train_csv}")
print(f"test_csv : {test_csv}")

In [ ]:
import json, time
from pathlib import Path
from ww_trainer.quickstart import train_from_wakeword

results_dir = Path(OUTPUT_DIR) / "micro_results"
results_dir.mkdir(parents=True, exist_ok=True)

all_results = []

for tier in MICRO_TIERS:
    result_file = results_dir / f"{tier}.json"
    model_subdir = Path(OUTPUT_DIR) / "models" / tier

    print(f"\n{'='*60}")
    print(f"Tier: {tier!r}")

    if SKIP_COMPLETED and result_file.exists():
        saved = json.loads(result_file.read_text())
        print(f"  SKIP (done): F1={saved.get('f1', 0):.4f}")
        all_results.append(saved)
        continue

    t0 = time.time()
    try:
        r = train_from_wakeword(
            WAKE_WORD,
            str(model_subdir),
            tier=tier,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            device=DEVICE,
            seed=SEED,
            reuse_dataset=True,
            **_aug_kwargs_full,
        )
        elapsed = time.time() - t0
        row = {
            "tier": tier,
            "f1": r.metrics.get("f1", 0.0),
            "precision": r.metrics.get("precision", 0.0),
            "recall": r.metrics.get("recall", 0.0),
            "elapsed_s": round(elapsed, 1),
            "head_onnx": str(r.best_onnx_path) if r.best_onnx_path else "",
            "feat_onnx": str(model_subdir / "model" / "best_f1_featurizer.onnx"),
            "status": "ok",
        }
        print(f"  DONE: F1={row['f1']:.4f}  ({elapsed:.0f}s)")
    except Exception as exc:
        elapsed = time.time() - t0
        row = {
            "tier": tier, "f1": 0.0, "precision": 0.0, "recall": 0.0,
            "elapsed_s": round(elapsed, 1),
            "head_onnx": "", "feat_onnx": "",
            "status": f"error: {exc}",
        }
        print(f"  ERROR: {exc}")

    result_file.write_text(json.dumps(row, indent=2))
    all_results.append(row)

print(f"\nAll micro tiers done. {sum(1 for r in all_results if r['status']=='ok')}/{len(all_results)} succeeded.")

In [ ]:
import pandas as pd
from pathlib import Path

# ── Size audit ────────────────────────────────────────────────────────────────
# For each tier: parameter count, ONNX head size, estimated int8 size

ESP32_BUDGETS = {
    "esp32_nano": 1,   # KB
    "esp32_sweet": 10,
    "esp32_max": 50,
    "micro": 512,
    "delta_micro": 512,
}

audit_rows = []
for row in all_results:
    tier = row["tier"]
    head_path = Path(row["head_onnx"]) if row["head_onnx"] else None
    feat_path = Path(row["feat_onnx"]) if row["feat_onnx"] else None

    head_kb = head_path.stat().st_size / 1024 if head_path and head_path.exists() else None
    feat_kb = feat_path.stat().st_size / 1024 if feat_path and feat_path.exists() else None
    int8_est_kb = round(head_kb / 4, 1) if head_kb else None
    budget_kb = ESP32_BUDGETS.get(tier)
    fits = (head_kb is not None and budget_kb is not None and head_kb <= budget_kb * 4)

    # Try to get param count from tier info
    try:
        from ww_trainer.tiers import get_tier
        tier_cfg = get_tier(tier)
        params = getattr(tier_cfg, "approx_params", "?")
    except Exception:
        params = "?"

    audit_rows.append({
        "tier": tier,
        "params": params,
        "head_onnx_kb": round(head_kb, 1) if head_kb else "missing",
        "feat_onnx_kb": round(feat_kb, 1) if feat_kb else "missing",
        "int8_est_kb": int8_est_kb if int8_est_kb else "?",
        "budget_kb": budget_kb,
        "within_budget": "YES" if fits else ("NO" if head_kb else "N/A"),
        "f1": row.get("f1", 0.0),
        "status": row["status"],
    })

df_audit = pd.DataFrame(audit_rows)
print("Size audit:")
print(df_audit.to_string(index=False))
print()
print(f"SIZE_BUDGET_KB={SIZE_BUDGET_KB} KB — tiers exceeding this (head ONNX):")
for r in audit_rows:
    kb = r["head_onnx_kb"]
    if isinstance(kb, float) and kb > SIZE_BUDGET_KB:
        print(f"  {r['tier']}: {kb:.1f} KB  (budget: {SIZE_BUDGET_KB} KB)")

In [ ]:
import time
import numpy as np
from pathlib import Path

# ── Latency benchmark ─────────────────────────────────────────────────────────
# measure_latency() runs N forward passes on random 1-second audio and
# returns mean latency in milliseconds.

try:
    from ww_trainer.benchmark import measure_latency
    _have_benchmark = True
except ImportError:
    _have_benchmark = False
    print("ww_trainer.benchmark not available — using manual timing")

latency_rows = []
for row in all_results:
    if row["status"] != "ok":
        latency_rows.append({"tier": row["tier"], "latency_ms": None, "rtf": None})
        continue
    tier = row["tier"]
    feat_path = Path(row["feat_onnx"])
    head_path = Path(row["head_onnx"])
    if not feat_path.exists() or not head_path.exists():
        latency_rows.append({"tier": tier, "latency_ms": None, "rtf": None})
        continue

    if _have_benchmark:
        try:
            lat_ms = measure_latency(str(feat_path), str(head_path), n_runs=20)
        except Exception as e:
            lat_ms = None
            print(f"  {tier}: benchmark error: {e}")
    else:
        # Manual fallback: time OnnxWakeWordInferencer.infer()
        from ww_trainer.inference import OnnxWakeWordInferencer
        inf = OnnxWakeWordInferencer(str(feat_path), str(head_path))
        dummy = np.random.randn(16000).astype(np.float32)
        # Warm up
        for _ in range(3):
            inf.infer(dummy)
        t0 = time.perf_counter()
        for _ in range(20):
            inf.infer(dummy)
        lat_ms = (time.perf_counter() - t0) / 20 * 1000

    # RTF: latency / audio_duration (1 second of audio)
    rtf = (lat_ms / 1000) if lat_ms is not None else None
    print(f"  {tier}: {lat_ms:.1f} ms  (RTF={rtf:.4f})" if lat_ms else f"  {tier}: n/a")
    latency_rows.append({"tier": tier, "latency_ms": round(lat_ms, 2) if lat_ms else None, "rtf": round(rtf, 5) if rtf else None})

df_latency = pd.DataFrame(latency_rows)
print()
print(df_latency.to_string(index=False))

In [ ]:
from pathlib import Path

# ── C header export for the best ESP32 tier ───────────────────────────────────
# export_to_c_header() writes a .h file with the model weights as a const array.
# The header can be included in ESP-IDF or Arduino projects.

esp32_export_dir = Path(OUTPUT_DIR) / "esp32_export"
esp32_export_dir.mkdir(parents=True, exist_ok=True)

# Find the result for the configured TIER
_c_export_row = next((r for r in all_results if r["tier"] == TIER and r["status"] == "ok"), None)
if _c_export_row is None:
    # Fall back to best available ESP32 tier
    for fallback in ["esp32_sweet", "esp32_nano", "esp32_max"]:
        _c_export_row = next((r for r in all_results if r["tier"] == fallback and r["status"] == "ok"), None)
        if _c_export_row:
            print(f"TIER={TIER!r} not available, falling back to {fallback!r}")
            TIER = fallback
            break

if _c_export_row:
    head_onnx = Path(_c_export_row["head_onnx"])
    c_header  = esp32_export_dir / "model.h"
    try:
        from ww_trainer.export_c import export_to_c_header
        export_to_c_header(
            head_onnx,
            c_header,
            model_name="ww_model",
            wake_word=WAKE_WORD,
        )
        size_kb = c_header.stat().st_size / 1024
        print(f"C header exported: {c_header}  ({size_kb:.1f} KB)")
        print(f"Wake word embedded in header: {WAKE_WORD!r}")
        print()
        # Print first 30 lines as preview
        lines = c_header.read_text().splitlines()[:30]
        print("--- model.h preview ---")
        for line in lines:
            print(line)
        print("...")
    except ImportError:
        print("ww_trainer.export_c not available — skipping C header export")
        print(f"Hint: implement export_c.export_to_c_header() in ww_trainer/export_c.py")
        print(f"Head ONNX for manual export: {head_onnx}")
else:
    print("No successful ESP32 tier found — cannot export C header.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ── Results table ─────────────────────────────────────────────────────────────

df_base = pd.DataFrame(all_results)
df_sz   = pd.DataFrame(audit_rows)
df_lat  = pd.DataFrame(latency_rows)

df = df_base.merge(df_sz[["tier","params","head_onnx_kb","int8_est_kb","within_budget"]],
                   on="tier", how="left")
df = df.merge(df_lat, on="tier", how="left")

# ESP32 fit flags
for esp_tier, budget_kb in [("esp32_nano", 5), ("esp32_sweet", 40), ("esp32_max", 60)]:
    col = f"fits_{esp_tier}"
    df[col] = df["head_onnx_kb"].apply(
        lambda kb: "YES" if isinstance(kb, (int, float)) and kb <= budget_kb else "NO"
    )

_cols = ["tier", "params", "head_onnx_kb", "int8_est_kb", "f1",
         "latency_ms", "fits_esp32_nano", "fits_esp32_sweet", "fits_esp32_max"]
_cols = [c for c in _cols if c in df.columns]

print("Micro tier results:")
print(df[_cols].to_string(index=False))

# Save CSV
csv_out = Path(OUTPUT_DIR) / "micro_results.csv"
df.to_csv(csv_out, index=False)
print(f"\nResults saved: {csv_out}")

# Bar chart
df_ok = df[df["status"] == "ok"].copy()
if not df_ok.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"Micro tiers — {WAKE_WORD!r}", fontsize=12)

    ax = axes[0]
    ax.bar(df_ok["tier"], df_ok["f1"], color="steelblue", edgecolor="white")
    ax.set_ylabel("F1")
    ax.set_title("F1 by tier")
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=30)

    ax2 = axes[1]
    if "head_onnx_kb" in df_ok.columns:
        sizes = pd.to_numeric(df_ok["head_onnx_kb"], errors="coerce")
        ax2.bar(df_ok["tier"], sizes, color="coral", edgecolor="white")
        ax2.set_ylabel("Head ONNX size (KB)")
        ax2.set_title("ONNX head size")
        ax2.tick_params(axis="x", rotation=30)
        for budget, label in [(1, "nano"), (10, "sweet"), (50, "max")]:
            ax2.axhline(budget, color="red", linestyle="--", linewidth=0.8, alpha=0.7)
            ax2.text(len(df_ok) - 0.5, budget + 0.5, f"ESP32 {label}", fontsize=7, color="red")

    plt.tight_layout()
    plot_path = str(Path(OUTPUT_DIR) / "micro_results.png")
    plt.savefig(plot_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {plot_path}")

In [ ]:
import csv
import numpy as np
import torchaudio
from pathlib import Path
from ww_trainer.inference import OnnxWakeWordInferencer

# ── Inference test on a positive sample ───────────────────────────────────────
# Use the best-F1 successful tier for the inference demo.

best_row = sorted(
    [r for r in all_results if r["status"] == "ok"],
    key=lambda r: r.get("f1", 0),
    reverse=True,
)
best_row = best_row[0] if best_row else None

if best_row is None:
    print("No successful tier — cannot run inference test.")
else:
    feat_onnx = Path(best_row["feat_onnx"])
    head_onnx = Path(best_row["head_onnx"])

    if feat_onnx.exists() and head_onnx.exists():
        inferencer = OnnxWakeWordInferencer(str(feat_onnx), str(head_onnx))

        # Find a positive sample
        _pos_path = None
        with open(test_csv) as f:
            for row in csv.reader(f):
                if len(row) >= 2 and row[1].strip() == "1" and Path(row[0]).exists():
                    _pos_path = row[0]
                    break

        if _pos_path:
            wav, sr = torchaudio.load(_pos_path)
            if sr != 16000:
                wav = torchaudio.functional.resample(wav, sr, 16000)
            wav_np = wav.mean(0).numpy().astype(np.float32)
            score = inferencer.infer(wav_np)
            print(f"Tier        : {best_row['tier']!r}  (F1={best_row['f1']:.4f})")
            print(f"Sample      : {Path(_pos_path).name}")
            print(f"Score       : {score:.4f}  ({'PASS ✓' if score > 0.5 else 'LOW — may need more training'})")
        else:
            print("No positive sample found in test CSV.")
    else:
        print(f"ONNX files missing for tier {best_row['tier']!r}")

print()
print("=" * 60)
print("CLI commands:")
if best_row and Path(best_row.get("feat_onnx", "")).exists():
    print(f"  .venv/bin/python scripts/eval/test_wakeword.py \\")
    print(f"      --featurizer {best_row['feat_onnx']} \\")
    print(f"      --model      {best_row['head_onnx']} \\")
    print(f"      --audio      sample.wav")
c_header = Path(OUTPUT_DIR) / "esp32_export" / "model.h"
if c_header.exists():
    print(f"\nC header: {c_header}")
    print("Include in firmware:  #include \"model.h\"")
print("=" * 60)